# DriveGuard AI: Driver Drowsiness Detection with Neural Networks
**ANN Lab Project** | Dataset: Driver Drowsiness Dataset (DDD, Kaggle) | Tools: Python, TensorFlow/Keras, NumPy, MiniSom, Streamlit

This notebook implements everything promised in the proposal:

| Proposal item | Where in this notebook |
|---|---|
| 3.2.1 Dataset, 3.2.2 EDA | Sections 2 and 3 |
| 3.2.3 Preprocessing, split, augmentation | Sections 4 and 8 |
| 3.2.4 Feature engineering (flattened vectors, CNN tensors) | Section 5 |
| 3.2.5 Models: Logistic Regression, Perceptron, MLP, CNN | Sections 6 to 9 |
| Overfitting control (dropout, early stopping, augmentation) | Section 9 |
| 3.4 Evaluation (accuracy, precision, recall, F1) | Section 10 |
| 3.2.5 Kohonen SOM | Section 11 |
| Streamlit interface | `app.py` (separate file) |

**How to run (Google Colab):** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**.
Do a quick dry run first by setting `MAX_PER_CLASS = 2000` in Section 1, then set it back to `None` for the real run.

## 1. Setup and configuration

In [ ]:
%pip install -q minisom kagglehub

In [ ]:
import os, re, random, time, json, shutil, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ------------------------------------------------------------------ CONFIG (edit here)
SEED          = 42
IMG_SIZE      = 64          # images are resized to IMG_SIZE x IMG_SIZE (proposal: 64 or 128)
EPOCHS        = 15          # max epochs per model (early stopping may stop sooner)
BATCH_SIZE    = 64
MAX_PER_CLASS = None        # None = use ALL images. Use e.g. 2000 for a fast dry run.
SPLIT_MODE    = "block"     # "block" = leakage-safe split (recommended), "random" = plain random split
BLOCK_SIZE    = 1000        # consecutive frames kept together in one split (see Section 4)
DATA_ROOT     = None        # None = download automatically. Or set a folder path yourself.
OUTPUT_DIR    = "outputs"
RUN_LEAKAGE_CHECK = True    # extra experiment: random split vs block split
# ----------------------------------------------------------------------------------------

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
CLASS_NAMES = ["Not Drowsy", "Drowsy"]      # label 0 = Not Drowsy, label 1 = Drowsy
print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Load the Driver Drowsiness Dataset (DDD)
The notebook tries to download DDD automatically with `kagglehub`. If that fails (for example no Kaggle login), use one of these:
* **Kaggle Notebook:** click *Add Input*, search "Driver Drowsiness Dataset (DDD)", then set `DATA_ROOT = "/kaggle/input/driver-drowsiness-dataset-ddd"`.
* **Colab + Google Drive:** download the dataset zip from Kaggle, upload it to Drive, then run `from google.colab import drive; drive.mount('/content/drive')` and unzip it, then set `DATA_ROOT` to the unzipped folder.
* **Your own PC:** set `DATA_ROOT = r"C:\path\to\Driver Drowsiness Dataset (DDD)"`.

The code finds the "Drowsy" and "Non Drowsy" folders by itself, wherever they are inside `DATA_ROOT`.

In [ ]:
if DATA_ROOT is None:
    try:
        import kagglehub
        DATA_ROOT = kagglehub.dataset_download("ismailnasri20/driver-drowsiness-dataset-ddd")
        print("Downloaded to:", DATA_ROOT)
    except Exception as e:
        raise SystemExit("Automatic download failed: %s\nSet DATA_ROOT manually (see the instructions above)." % e)

IMG_EXT = (".png", ".jpg", ".jpeg")

def find_class_dirs(root):
    """Walk the dataset folder and return {label: [folders]} for the drowsy / non-drowsy folders."""
    found = {0: [], 1: []}
    for dirpath, _, filenames in os.walk(root):
        imgs = [f for f in filenames if f.lower().endswith(IMG_EXT)]
        if len(imgs) < 50:
            continue
        name = os.path.basename(dirpath).lower().replace("_", " ")
        if any(k in name for k in ("non", "not", "alert", "awake")):
            found[0].append(dirpath)
        elif "drows" in name:
            found[1].append(dirpath)
    return found

class_dirs = find_class_dirs(DATA_ROOT)
for lab in (0, 1):
    print(CLASS_NAMES[lab], "->", class_dirs[lab])
assert class_dirs[0] and class_dirs[1], "Could not find the two class folders. Check DATA_ROOT."

# list files in sorted order (frames of one video stay next to each other in this order)
paths, labels, positions = [], [], []
for lab in (0, 1):
    files = []
    for d in class_dirs[lab]:
        files += [os.path.join(d, f) for f in sorted(os.listdir(d)) if f.lower().endswith(IMG_EXT)]
    n_total = len(files)
    if MAX_PER_CLASS is not None and n_total > MAX_PER_CLASS:
        keep = np.linspace(0, n_total - 1, MAX_PER_CLASS).astype(int)   # evenly spaced, keeps the video structure
    else:
        keep = np.arange(n_total)
    paths += [files[i] for i in keep]
    labels += [lab] * len(keep)
    positions += list(keep)
    print(f"{CLASS_NAMES[lab]}: {n_total} images found, using {len(keep)}")

def load_one(p):
    img = cv2.imread(p)
    if img is None:
        return None                                   # corrupted / unreadable image
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    loaded = list(pool.map(load_one, paths))
ok = [i for i, im in enumerate(loaded) if im is not None]
print(f"Loaded {len(ok)} images, removed {len(loaded) - len(ok)} unreadable ones in {time.time() - t0:.0f}s")

X = np.stack([loaded[i] for i in ok]).astype("uint8")          # (N, IMG_SIZE, IMG_SIZE, 3)
y = np.array([labels[i] for i in ok], dtype="int32")
pos = np.array([positions[i] for i in ok])
del loaded
print("X:", X.shape, "| y:", y.shape, "| drowsy share: %.1f%%" % (100 * y.mean()))

## 3. Exploratory Data Analysis (proposal 3.2.2)
Class balance, sample images, mean images, pixel histograms, and a **frame similarity check**.
The last check matters: DDD images are frames cut from videos, so neighbouring frames are almost identical.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
counts = pd.Series(y).map({0: CLASS_NAMES[0], 1: CLASS_NAMES[1]}).value_counts()
sns.barplot(x=counts.index, y=counts.values, ax=axes[0], palette=["#4C72B0", "#C44E52"])
axes[0].set_title("Class balance"); axes[0].set_ylabel("Number of images")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, str(v), ha="center", va="bottom")
axes[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%", colors=["#4C72B0", "#C44E52"])
axes[1].set_title("Class share")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/eda_class_balance.png", dpi=150); plt.show()

# sample images from each class
rng = np.random.RandomState(SEED)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for r, lab in enumerate((0, 1)):
    idx = rng.choice(np.where(y == lab)[0], 8, replace=False)
    for c, i in enumerate(idx):
        axes[r, c].imshow(X[i]); axes[r, c].axis("off")
    axes[r, 0].set_title(CLASS_NAMES[lab], loc="left", fontsize=11)
plt.suptitle("Random samples from each class"); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/eda_samples.png", dpi=150); plt.show()

# mean image per class and pixel intensity histogram
sub = rng.choice(len(X), min(6000, len(X)), replace=False)          # subsample keeps RAM low
gray_sub, y_sub = X[sub].mean(axis=3), y[sub]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for lab in (0, 1):
    axes[lab].imshow(X[y == lab].mean(axis=0).astype("uint8")); axes[lab].axis("off")
    axes[lab].set_title(f"Mean image: {CLASS_NAMES[lab]}")
for lab, col in ((0, "#4C72B0"), (1, "#C44E52")):
    axes[2].hist(gray_sub[y_sub == lab].ravel()[::20], bins=50, alpha=0.55, color=col, label=CLASS_NAMES[lab], density=True)
axes[2].set_title("Pixel intensity histogram"); axes[2].set_xlabel("Intensity (0-255)"); axes[2].legend()
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/eda_mean_hist.png", dpi=150); plt.show()

# frame similarity check: are neighbouring frames nearly identical?
order = np.argsort(pos + 10_000_000 * y)           # sort by class then by original frame order
Xs = X[order[:4000]].astype("float32")                # first 4000 frames in video order (keeps RAM low)
adjacent = np.abs(Xs[1:] - Xs[:-1]).mean(axis=(1, 2, 3))
rand_pairs = np.abs(Xs[rng.permutation(len(Xs))] - Xs).mean(axis=(1, 2, 3))
print(f"Mean pixel difference between ADJACENT frames : {adjacent.mean():.2f}")
print(f"Mean pixel difference between RANDOM images   : {rand_pairs.mean():.2f}")
print("If the first number is much smaller, neighbouring frames are near-duplicates, so a random split would leak "
      "almost identical images into both training and test sets. Section 4 avoids this.")
del Xs

## 4. Train / validation / test split (70 / 15 / 15)
**Why not a plain random split?** DDD frames come from videos of the same drivers. With a random split, a test image
often has a twin (the previous or next frame) in the training set, so the scores look unrealistically high.
The default `SPLIT_MODE = "block"` keeps **blocks of consecutive frames together**, so each block lands in only one split.
The dataset has no driver IDs, so this is an approximation (blocks may still share a driver in rare cases), and the report should say so.

In [ ]:
def make_split(y, pos, mode="block"):
    n = len(y)
    if mode == "random":
        idx = np.arange(n)
        tr, tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
        va, te = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=SEED)
        return tr, va, te
    groups = np.array([f"{c}_{p // BLOCK_SIZE}" for c, p in zip(y, pos)])
    rng = np.random.RandomState(SEED)
    assign = {}
    for c in (0, 1):
        gs = np.unique(groups[y == c]); rng.shuffle(gs); g = len(gs)
        n_va = max(1, int(round(0.15 * g)))
        n_te = max(1, int(round(0.15 * g)))
        n_tr = g - n_va - n_te
        assert n_tr >= 1, "Too few blocks. Lower BLOCK_SIZE or use more images."
        for i, name in enumerate(gs):
            assign[name] = "train" if i < n_tr else ("val" if i < n_tr + n_va else "test")
    part = np.array([assign[g] for g in groups])
    return np.where(part == "train")[0], np.where(part == "val")[0], np.where(part == "test")[0]

tr_idx, va_idx, te_idx = make_split(y, pos, SPLIT_MODE)
X_train, y_train = X[tr_idx], y[tr_idx]
X_val,   y_val   = X[va_idx], y[va_idx]
X_test,  y_test  = X[te_idx], y[te_idx]
for name, yy in (("Train", y_train), ("Validation", y_val), ("Test", y_test)):
    print(f"{name:10s}: {len(yy):6d} images | drowsy share {100 * yy.mean():.1f}%")

## 5. Feature engineering (proposal 3.2.4)
* **Logistic regression, perceptron, MLP:** grayscale, scaled to 0 to 1 (`x' = x / 255`), flattened into one vector.
* **CNN:** the RGB image tensor is kept. Scaling by 1/255 happens inside the model (a `Rescaling` layer), so the Streamlit app can pass raw pixels.

In [ ]:
def to_flat_gray(images, chunk=4096):
    w = np.array([0.299, 0.587, 0.114], dtype="float32")
    out = np.empty((len(images), IMG_SIZE * IMG_SIZE), dtype="float32")
    for s in range(0, len(images), chunk):                                               # chunked to save RAM
        gray = images[s:s + chunk].astype("float32") @ w                                 # RGB -> gray
        out[s:s + chunk] = (gray / 255.0).reshape(len(gray), -1)                         # x' = x / 255
    return out

F_train, F_val, F_test = to_flat_gray(X_train), to_flat_gray(X_val), to_flat_gray(X_test)
D = F_train.shape[1]
print("Flat feature vectors:", F_train.shape, "| CNN tensors:", X_train.shape)

## 6. Helper functions: training, evaluation and plots

In [ ]:
results = []          # one row per model / variant
histories = {}        # training curves
probs_store = {}      # predicted probabilities on val / test for each model

def metrics_row(name, y_true, y_pred):
    return dict(accuracy=accuracy_score(y_true, y_pred),
                precision=precision_score(y_true, y_pred, zero_division=0),
                recall=recall_score(y_true, y_pred, zero_division=0),
                f1=f1_score(y_true, y_pred, zero_division=0))

def register(name, y_val_prob, y_test_prob, n_params=None, train_time=None, epochs_run=None):
    """Store validation + test metrics for a model (probabilities in [0,1], threshold 0.5)."""
    v = metrics_row(name, y_val,  (y_val_prob  > 0.5).astype(int))
    t = metrics_row(name, y_test, (y_test_prob > 0.5).astype(int))
    results.append({"model": name,
                    "val_f1": v["f1"], "test_accuracy": t["accuracy"], "test_precision": t["precision"],
                    "test_recall": t["recall"], "test_f1": t["f1"],
                    "epochs_run": epochs_run, "params": n_params, "train_s": train_time})
    probs_store[name] = (y_val_prob, y_test_prob)
    print(f"{name:34s} | val F1 {v['f1']:.4f} | test acc {t['accuracy']:.4f} | P {t['precision']:.4f} | "
          f"R {t['recall']:.4f} | F1 {t['f1']:.4f}")

def train_keras(model, Xtr, ytr, Xva, yva, early_stop=False, epochs=EPOCHS, batch=BATCH_SIZE, patience=3):
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)] if early_stop else []
    t0 = time.time()
    h = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=epochs, batch_size=batch, callbacks=cbs, verbose=2)
    return h.history, time.time() - t0

def plot_curves(hist, title, fname):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    axes[0].plot(hist["loss"], label="train"); axes[0].plot(hist["val_loss"], label="validation")
    axes[0].set_title(f"{title}: loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(hist["accuracy"], label="train"); axes[1].plot(hist["val_accuracy"], label="validation")
    axes[1].set_title(f"{title}: accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/{fname}", dpi=150); plt.show()

## 7. Model 1 and 2: Logistic Regression (baseline) and Single-Layer Perceptron
* **Logistic regression** = one neuron with sigmoid output, `P(drowsy) = 1 / (1 + e^-(w.x + b))`, trained with binary cross-entropy.
* **Single-layer perceptron** is written **from scratch in NumPy** with the perceptron learning rule: `w <- w + lr * (y - y_hat) * x`
  (no hidden layer, step activation, linear decision boundary).

In [ ]:
# ---------------- Model 1: Logistic regression (single sigmoid neuron)
logreg = keras.Sequential([layers.Input((D,)), layers.Dense(1, activation="sigmoid")], name="logistic_regression")
logreg.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
h, secs = train_keras(logreg, F_train, y_train, F_val, y_val, early_stop=True)
histories["Logistic Regression"] = h
plot_curves(h, "Logistic Regression", "curves_logreg.png")
register("Logistic Regression", logreg.predict(F_val, verbose=0).ravel(), logreg.predict(F_test, verbose=0).ravel(),
         logreg.count_params(), secs, len(h["loss"]))

In [ ]:
# ---------------- Model 2: Single-layer perceptron, from scratch (NumPy)
class Perceptron:
    """Perceptron learning rule with mini-batch updates. Inputs are mean-centred internally."""
    def __init__(self, lr=0.01, epochs=EPOCHS, batch=128, seed=SEED):
        self.lr, self.epochs, self.batch, self.seed = lr, epochs, batch, seed
    def score(self, X):
        return (X - self.mu) @ self.w + self.b
    def predict(self, X):
        return (self.score(X) > 0).astype(int)
    def predict_proba(self, X):
        return 1 / (1 + np.exp(-np.clip(self.score(X), -30, 30)))     # only used to report a pseudo-probability
    def fit(self, X, y, Xv, yv):
        rng = np.random.RandomState(self.seed)
        self.mu = X.mean(axis=0); self.w = np.zeros(X.shape[1], dtype="float32"); self.b = 0.0
        hist = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
        best = (-1, None)
        for ep in range(self.epochs):
            idx = rng.permutation(len(X))
            for s in range(0, len(X), self.batch):
                b = idx[s:s + self.batch]
                err = y[b] - self.predict(X[b])                        # +1, 0 or -1
                self.w += self.lr * ((X[b] - self.mu).T @ err) / len(b)
                self.b += self.lr * err.mean()
            tr_err = float(np.mean(self.predict(X) != y)); va_err = float(np.mean(self.predict(Xv) != yv))
            hist["loss"].append(tr_err); hist["val_loss"].append(va_err)          # "loss" = misclassification rate
            hist["accuracy"].append(1 - tr_err); hist["val_accuracy"].append(1 - va_err)
            if 1 - va_err > best[0]:
                best = (1 - va_err, (self.w.copy(), self.b))
        self.w, self.b = best[1]                                       # keep the best epoch (validation-based)
        return hist

t0 = time.time()
perc = Perceptron()
h = perc.fit(F_train, y_train, F_val, y_val)
histories["Single-Layer Perceptron"] = h
plot_curves(h, "Single-Layer Perceptron (loss = error rate)", "curves_perceptron.png")
register("Single-Layer Perceptron", perc.predict_proba(F_val), perc.predict_proba(F_test),
         D + 1, time.time() - t0, len(h["loss"]))

## 8. Model 3: Multi-Layer Perceptron (MLP)
Hidden layers with **ReLU**, sigmoid output, **backpropagation** with the **Adam** optimizer and **binary cross-entropy** loss.
We train it twice: a plain version and a version with dropout and early stopping (overfitting control).

In [ ]:
def build_mlp(dropout=False):
    m = keras.Sequential(name="mlp_dropout" if dropout else "mlp_plain")
    m.add(layers.Input((D,)))
    m.add(layers.Dense(256, activation="relu"))
    if dropout: m.add(layers.Dropout(0.4))
    m.add(layers.Dense(128, activation="relu"))
    if dropout: m.add(layers.Dropout(0.4))
    m.add(layers.Dense(1, activation="sigmoid"))
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m

mlp_plain = build_mlp(False)
h, secs = train_keras(mlp_plain, F_train, y_train, F_val, y_val, early_stop=False)
histories["MLP (plain)"] = h; plot_curves(h, "MLP (plain)", "curves_mlp_plain.png")
register("MLP (plain)", mlp_plain.predict(F_val, verbose=0).ravel(), mlp_plain.predict(F_test, verbose=0).ravel(),
         mlp_plain.count_params(), secs, len(h["loss"]))

mlp_reg = build_mlp(True)
h, secs = train_keras(mlp_reg, F_train, y_train, F_val, y_val, early_stop=True)
histories["MLP (dropout + early stopping)"] = h; plot_curves(h, "MLP (dropout + early stopping)", "curves_mlp_reg.png")
register("MLP (dropout + early stopping)", mlp_reg.predict(F_val, verbose=0).ravel(), mlp_reg.predict(F_test, verbose=0).ravel(),
         mlp_reg.count_params(), secs, len(h["loss"]))

## 9. Model 4: CNN and the overfitting-control experiment
The CNN has 3 convolution + pooling blocks, a 64-unit feature layer (named `features`, used later by the SOM) and a sigmoid output.
Four versions are trained to measure each overfitting-control technique:

| Variant | Dropout | Early stopping | Augmentation |
|---|---|---|---|
| CNN (plain) | no | no | no |
| CNN (+dropout) | yes | no | no |
| CNN (+dropout +early stopping) | yes | yes | no |
| CNN (+dropout +early stopping +augmentation) | yes | yes | yes |

Augmentation = random horizontal flip, small rotation and brightness change (training only; Keras switches these layers off at test time).

In [ ]:
def build_cnn(dropout=False, augment=False, name="cnn"):
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1.0 / 255)(inp)                                  # x' = x / 255 inside the model
    if augment:
        x = layers.RandomFlip("horizontal")(x)
        x = layers.RandomRotation(0.05)(x)
        x = layers.RandomBrightness(0.2, value_range=(0, 1))(x)
    for filters in (32, 64, 128):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        if dropout: x = layers.Dropout(0.2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation="relu", name="features")(x)
    if dropout: x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m = keras.Model(inp, out, name=name)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m

cnn_variants = [
    ("CNN (plain)",                                 dict(dropout=False, augment=False), False),
    ("CNN (+dropout)",                              dict(dropout=True,  augment=False), False),
    ("CNN (+dropout +early stopping)",              dict(dropout=True,  augment=False), True),
    ("CNN (+dropout +early stopping +augmentation)", dict(dropout=True,  augment=True),  True),
]
cnn_models = {}
for name, kw, es in cnn_variants:
    print("\n=== Training:", name)
    keras.backend.clear_session(); tf.random.set_seed(SEED)
    m = build_cnn(name=name.replace(" ", "_").replace("+", "").replace("(", "").replace(")", ""), **kw)
    h, secs = train_keras(m, X_train, y_train, X_val, y_val, early_stop=es)
    histories[name] = h
    plot_curves(h, name, "curves_" + re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_") + ".png")
    register(name, m.predict(X_val, verbose=0).ravel(), m.predict(X_test, verbose=0).ravel(),
             m.count_params(), secs, len(h["loss"]))
    cnn_models[name] = m

## 10. Evaluation and model comparison (proposal 3.4)
Accuracy, precision, recall and F1 on the **unseen test set**. The best model is chosen by **validation F1** (never by test score).

In [ ]:
res = pd.DataFrame(results)
res_show = res.copy()
for c in ("val_f1", "test_accuracy", "test_precision", "test_recall", "test_f1"):
    res_show[c] = res_show[c].round(4)
res_show["train_s"] = res_show["train_s"].round(0)
display(res_show)
res.to_csv(f"{OUTPUT_DIR}/results_table.csv", index=False)

# bar chart of the four metrics
plot_df = res.melt(id_vars="model", value_vars=["test_accuracy", "test_precision", "test_recall", "test_f1"],
                   var_name="metric", value_name="score")
plt.figure(figsize=(13, 4.8))
sns.barplot(data=plot_df, x="model", y="score", hue="metric")
plt.xticks(rotation=25, ha="right"); plt.ylim(0, 1.05); plt.title("Test-set comparison of all models")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/model_comparison.png", dpi=150); plt.show()

# confusion matrices
names = list(res["model"])
cols = 4; rows = int(np.ceil(len(names) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.6 * rows)); axes = np.array(axes).ravel()
for ax, n in zip(axes, names):
    cm = confusion_matrix(y_test, (probs_store[n][1] > 0.5).astype(int), labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(n, fontsize=8); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
for ax in axes[len(names):]: ax.axis("off")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/confusion_matrices.png", dpi=150); plt.show()

# pick the best CNN by validation F1
cnn_rows = res[res["model"].str.startswith("CNN")]
best_name = cnn_rows.sort_values("val_f1", ascending=False).iloc[0]["model"]
best_cnn = cnn_models[best_name]
print("Best CNN (chosen on validation F1):", best_name)
print(classification_report(y_test, (probs_store[best_name][1] > 0.5).astype(int), target_names=CLASS_NAMES, digits=4))

## 11. Kohonen Self-Organizing Map (SOM)
The SOM is **unsupervised**: it never sees the labels while training. We feed it the 64-dimensional activations of the best CNN's `features` layer,
then check afterwards whether drowsy and alert samples land in different regions of the map.
Reported: quantization error, a U-matrix, a label map, and **cluster purity** (how well the map's neurons separate the two classes on unseen test images).

In [ ]:
from minisom import MiniSom

feat_model = keras.Model(best_cnn.input, best_cnn.get_layer("features").output)
rs = np.random.RandomState(SEED)
tr_sub = rs.choice(len(X_train), min(5000, len(X_train)), replace=False)
te_sub = rs.choice(len(X_test),  min(3000, len(X_test)),  replace=False)
Z_tr = feat_model.predict(X_train[tr_sub], verbose=0); yz_tr = y_train[tr_sub]
Z_te = feat_model.predict(X_test[te_sub],  verbose=0); yz_te = y_test[te_sub]
mu, sd = Z_tr.mean(0), Z_tr.std(0) + 1e-6
Z_tr, Z_te = (Z_tr - mu) / sd, (Z_te - mu) / sd                      # z-score the features

GRID = 10
som = MiniSom(GRID, GRID, Z_tr.shape[1], sigma=2.0, learning_rate=0.5, random_seed=SEED)
som.random_weights_init(Z_tr)
som.train_random(Z_tr, 8000)                                         # update rule: w <- w + lr(t) * h(t) * (x - w)

print("Quantization error  (train): %.4f" % som.quantization_error(Z_tr))
print("Quantization error  (test) : %.4f" % som.quantization_error(Z_te))

# label map: fraction of drowsy training samples that win each neuron
drowsy_frac = np.full((GRID, GRID), np.nan); count = np.zeros((GRID, GRID))
for x, lab in zip(Z_tr, yz_tr):
    i, j = som.winner(x); count[i, j] += 1
    drowsy_frac[i, j] = (0 if np.isnan(drowsy_frac[i, j]) else drowsy_frac[i, j]) + lab
drowsy_frac = drowsy_frac / np.where(count == 0, np.nan, count)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(som.distance_map().T, cmap="bone_r", origin="lower")
axes[0].set_title("U-matrix (dark = cluster borders)"); plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(drowsy_frac.T, cmap="coolwarm", vmin=0, vmax=1, origin="lower")
axes[1].set_title("Label map: share of drowsy samples per neuron\n(blue = alert, red = drowsy, white = empty)")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/som_maps.png", dpi=150); plt.show()

# purity on UNSEEN test images: label each neuron by its majority training label, then classify test samples
neuron_label = np.where(np.nan_to_num(drowsy_frac, nan=0.5) > 0.5, 1, 0)
filled = np.argwhere(count > 0)
for i in range(GRID):
    for j in range(GRID):
        if count[i, j] == 0:                                          # empty neuron: copy the nearest labelled neuron
            k = np.argmin(((filled - [i, j]) ** 2).sum(axis=1))
            neuron_label[i, j] = neuron_label[tuple(filled[k])]
train_purity = np.mean([neuron_label[som.winner(x)] == l for x, l in zip(Z_tr, yz_tr)])
print("SOM cluster purity on training samples: %.4f" % train_purity)
pred = np.array([neuron_label[som.winner(x)] for x in Z_te])
purity = (pred == yz_te).mean()
print("SOM cluster purity on unseen test images: %.4f" % purity)
print("(The SOM was trained without labels. A purity near 1.0 means drowsy and alert faces form separate regions; near 0.5 means they overlap.)")

## 12. Honest-evaluation check: random split vs block split (optional)
Re-trains the best CNN recipe on a **plain random split** and compares it with the block split. A big gap proves that near-duplicate frames inflate scores,
which is a valuable point for the report's discussion section.

In [ ]:
if RUN_LEAKAGE_CHECK:
    other_mode = "random" if SPLIT_MODE == "block" else "block"
    a, b, c = make_split(y, pos, other_mode)
    keras.backend.clear_session(); tf.random.set_seed(SEED)
    m = build_cnn(dropout=True, augment=False, name="cnn_leak_check")
    h, secs = train_keras(m, X[a], y[a], X[b], y[b], early_stop=True)
    p = (m.predict(X[c], verbose=0).ravel() > 0.5).astype(int)
    alt = metrics_row("alt", y[c], p)
    main = res[res["model"] == "CNN (+dropout +early stopping)"].iloc[0]
    cmp = pd.DataFrame({"split": [SPLIT_MODE, other_mode],
                        "test_accuracy": [main["test_accuracy"], alt["accuracy"]],
                        "test_f1": [main["test_f1"], alt["f1"]]})
    display(cmp.round(4))
    cmp.to_csv(f"{OUTPUT_DIR}/split_comparison.csv", index=False)
    print(f"Same CNN recipe, different split: test F1 = {alt['f1']:.4f} on the {other_mode} split vs {main['test_f1']:.4f} on the {SPLIT_MODE} split.")
else:
    print("Skipped (set RUN_LEAKAGE_CHECK = True to run it).")

## 13. Save the model for the Streamlit app
Saves `cnn_best.keras` and `model_meta.json` into the `outputs` folder, and zips everything for download.
Copy `cnn_best.keras` next to `app.py` on your computer, then run `streamlit run app.py`.

In [ ]:
best_cnn.save(f"{OUTPUT_DIR}/cnn_best.keras")
meta = {"img_size": IMG_SIZE, "class_names": CLASS_NAMES, "best_variant": best_name,
        "threshold": 0.5, "tensorflow_version": tf.__version__,
        "test_accuracy": float(res[res["model"] == best_name]["test_accuracy"].iloc[0]),
        "test_f1": float(res[res["model"] == best_name]["test_f1"].iloc[0])}
json.dump(meta, open(f"{OUTPUT_DIR}/model_meta.json", "w"), indent=2)
print(meta)

shutil.make_archive("driveguard_outputs", "zip", OUTPUT_DIR)
try:
    from google.colab import files
    files.download("driveguard_outputs.zip")
except Exception:
    print("Zip saved as driveguard_outputs.zip (download it from the file panel).")

## 14. What to write in the report (quick guide)
1. **Results table** from Section 10 and the bar chart: how performance changes from the linear baseline to the MLP and the CNN.
2. **Overfitting control**: compare the four CNN variants and their training/validation curves (Section 9).
3. **SOM**: the U-matrix, label map, quantization error and purity (Section 11).
4. **Split discussion**: why block split, plus the random-vs-block comparison (Section 12).
5. **Limitations**: frames come from a small number of drivers, no driver IDs, controlled lighting, face-crop images only, not a certified safety device.
6. **Future work**: LSTM over frame sequences, distraction and seatbelt modules, fusion risk score (as listed in the proposal).